In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../data/df_all_data.csv')

df

In [ ]:
df.columns

In [ ]:
# 1. Definir X e y
X = df.drop(columns=[
    'churned','user_id','birth_year','transaction_id',
    'create_date_notification','create_date_transaction',
    'create_date_user', 'ea_merchant_city', 'inactive_120_days', 'inactive_60_days', 'inactive_90_days', 'quarter'
]).copy()
y = df['churned']

display(X, y)

In [ ]:

# 2. Separar en train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=30)

display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# 3. Elegir muestra alineada de 1000 datos
sample_idx = X_train.sample(n=1000, random_state=42).index
X_train_sample = X_train.loc[sample_idx]
y_train_sample = y_train.loc[sample_idx]


In [ ]:
# 4. Identificar columnas numéricas y categóricas
num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes(exclude='number').columns


In [ ]:
# 5. Pipeline para columnas numéricas
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])


In [ ]:
# 6. Pipeline para columnas categóricas
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


In [ ]:
# 7. ColumnTransformer que aplica ambos pipelines
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])


In [ ]:
# 8. Pipeline completo con preprocesamiento + modelo
model_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression())
])


In [ ]:
# 9. Entrenar el pipeline
model_pipeline.fit(X_train_sample, y_train_sample)


In [ ]:
df['notification_count'].mean(), df['diff_days'].mean(), df['transaction_count'].mean()

In [ ]:
# 10. Predecir sobre nuevos datos
# Asegúrate que df_new tenga las mismas columnas que X (antes de limpieza)

data = {
    'country': ['PL'],
    'city': ['Gdansk'],
    'user_settings_crypto_unlocked': [1],
    'plan': ['STANDARD'],
    'attributes_notifications_marketing_push': [1.0],
    'attributes_notifications_marketing_email': [1.0],
    'num_contacts': [3],
    'num_referrals': [0],
    'num_successful_referrals': [0],
    'brand_device': ['Apple'],
    'reason': ['FIFTH_PAYMENT_PROMO'],
    'channel': ['EMAIL'],
    'status': ['SENT'],
    'transactions_type': ['CARD_PAYMENT'],
    'transactions_currency': ['EUR'],
    'amount_usd': [11.49],
    'transactions_state': ['COMPLETED'],
    'ea_cardholderpresence': ['FALSE'],
    'ea_merchant_mcc': [5441.0],
    'ea_merchant_country': ['GRC'],
    'direction': ['OUTBOUND'],
    'received_notification': [True],
    'age': [36],
    'age_group': ['35–44'],
    'notification_count': [30],
    'diff_days': [80],
    'transaction_count': [30]
}
# Create DataFrame
df_new = pd.DataFrame(data)


In [ ]:
df_new

In [ ]:
preds = model_pipeline.predict(df_new)
preds